# Phase 1: Spurgeon Continued Pretraining — Step 9: Evaluation & Export (Notebook C)

This notebook loads the trained Phase 1 PEFT QLoRA adapter checkpoints, evaluates its perplexity score on the 50-sermon holdout dataset, runs qualitative style validations on test prompts, and exports the final adapter weights.

## 1. Install Dependencies

In [1]:
# Install Unsloth and specific patched versions for Kaggle environment
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-g60vyl8u/unsloth_cb6dd8871ad34d90b147e8e1907a186e
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-g60vyl8u/unsloth_cb6dd8871ad34d90b147e8e1907a186e
  Resolved https://github.com/unslothai/unsloth.git to commit 2b76c6855a023d1bdd492c75193f83f66fe47562
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 90.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 44.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 78.0 MB/s eta 0:00:00
   ━━━

## 2. Load Model and LoRA Adapter

In [2]:
from unsloth import FastLanguageModel
import torch

# Set the path pointing to the mounted output of Run 2 (checkpoint-432)
LORA_PATH = "/kaggle/input/datasets/rafaelvieira1/spurgeon-training-run-1/checkpoints/checkpoint-432"
MAX_SEQ_LENGTH = 2048

# Load base model + LoRA adapter in 4-bit quantization natively
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = LORA_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = None,
    load_in_4bit   = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.6.6: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.56G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/171 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/617 [00:00<?, ?B/s]

unsloth/qwen2.5-3b-unsloth-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.6.6 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


## 3. Perplexity Evaluation on Holdout Set

In [3]:
import math
from datasets import load_from_disk

# Set model to inference mode
FastLanguageModel.for_inference(model)

# Load holdout dataset built in Notebook A
holdout_dataset = load_from_disk("/kaggle/input/datasets/rafaelvieira1/spurgeon-cpt-dataset/spurgeon_holdout_dataset")

total_loss = 0.0
total_tokens = 0

print("Evaluating holdout perplexity...")
for idx, doc in enumerate(holdout_dataset):
    # Tokenize sequence
    inputs = tokenizer(doc["text"], return_tensors="pt")
    inputs = {k: v.to("cuda") for k, v in inputs.items()}
    num_tokens = inputs["input_ids"].size(1)
    
    # Truncate sequence safely to fit context limits without VRAM spikes
    if num_tokens > MAX_SEQ_LENGTH:
        inputs = {k: v[:, :MAX_SEQ_LENGTH] for k, v in inputs.items()}
        num_tokens = MAX_SEQ_LENGTH
        
    with torch.no_grad():
        outputs = model(**inputs, labels=inputs["input_ids"])
        loss = outputs.loss.item()
        
    total_loss += loss * num_tokens
    total_tokens += num_tokens

average_loss = total_loss / total_tokens
perplexity = math.exp(average_loss)

print(f"\nEvaluation Results:")
print(f"  - Total Holdout Tokens: {total_tokens:,}")
print(f"  - Length-Weighted Loss: {average_loss:.4f}")
print(f"  - Holdout Perplexity: {perplexity:.2f}")

`use_return_dict` is deprecated! Use `return_dict` instead!


Evaluating holdout perplexity...


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API i


Evaluation Results:
  - Total Holdout Tokens: 102,400
  - Length-Weighted Loss: 2.2711
  - Holdout Perplexity: 9.69


## 4. Qualitative Style Validation

In [4]:
prompts = [
    "The love of Christ is not a cold, speculative thing. It is ",
    "Text: Romans 8:28. 'And we know that all things work together for good to them that love God.' My dear friends, ",
    "What, then, is saving faith? Let us examine this question carefully, for "
]

print("Running qualitative generations...\n")
for i, prompt in enumerate(prompts):
    print(f"--- Prompt {i+1} ---")
    print(f"Input: {prompt}")
    
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens=150, 
            temperature=0.7, 
            top_p=0.9, 
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
        
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output:\n{generated_text}\n")

Running qualitative generations...

--- Prompt 1 ---
Input: The love of Christ is not a cold, speculative thing. It is 


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
Both `max_new_tokens` (=1

Output:
The love of Christ is not a cold, speculative thing. It is 100% personal, emotional, and passionate. It is an intense love that has no limits. It is a love that will go to any lengths to be with us and to know us. It is a love that will risk everything to be with us. It is a love that will sacrifice everything for us. It is a love that will fight to the death for us. It is a love that will give up everything for us. It is a love that will put everything on the line for us.
This is the love of Christ. This is the love of Christ, a love that is not to be resisted. This is the love of Christ, a love that is not to be put in a corner. This is the love of

--- Prompt 2 ---
Input: Text: Romans 8:28. 'And we know that all things work together for good to them that love God.' My dear friends, 


Both `max_new_tokens` (=150) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output:
Text: Romans 8:28. 'And we know that all things work together for good to them that love God.' My dear friends, 1 know that some of you have been very greatly troubled, and I am not surprised that you should be so, for we are now in a time of trouble and distress. You know the world is in a state of great confusion. There is no peace, but only war, and the war is very dreadful. And the Lord has allowed the enemies of the Church of God to make great inroads upon us, and to break down the work of His Grace. We have been brought to the very edge of the precipice, and if we do not take care, we shall be dashed to pieces. We have been in the midst of the very jaws of death, and we have been in danger of being devoured. I know that you have

--- Prompt 3 ---
Input: What, then, is saving faith? Let us examine this question carefully, for 
Output:
What, then, is saving faith? Let us examine this question carefully, for 1) it is an important doctrine, and 2) it is a very simple one. It 

## 5. Save Final PEFT Adapter

In [5]:
output_path = "/kaggle/working/spurgeon_lora_final"
print(f"Saving final LoRA adapter to {output_path}...")
model.save_pretrained(output_path)
tokenizer.save_pretrained(output_path)
print("Weights saved successfully. Phase 1 pretraining evaluation and export completed.")


# ─── ADD THIS to Notebook C, after the LoRA save (cell 5) ───
print("Saving merged model in HuggingFace safetensors format...")
model.save_pretrained_merged(
    "/kaggle/working/spurgeon_phase1_merged_hf",
    tokenizer,
    save_method="merged_16bit",   # dequantizes + merges LoRA into base weights
)
print("Done. Save this output as a Kaggle dataset.")

Saving final LoRA adapter to /kaggle/working/spurgeon_lora_final...


Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/spurgeon_lora_final/tokenizer_config.json.


Weights saved successfully. Phase 1 pretraining evaluation and export completed.
Saving merged model in HuggingFace safetensors format...


config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in /kaggle/working/spurgeon_phase1_merged_hf/tokenizer_config.json.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [00:27<00:00, 13.91s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [00:41<00:00, 20.98s/it]


Unsloth: Merge process complete. Saved to `/kaggle/working/spurgeon_phase1_merged_hf`
Done. Save this output as a Kaggle dataset.


## 6. Convert and Export to GGUF (Original 16-bit)

We merge our PEFT adapter weights back into the base Qwen model and export directly to GGUF format. We specify `quantization_method = "f16"` to preserve the original 16-bit floating point precision of the weights without any quantization loss.

In [6]:
# Convert to GGUF format locally on Kaggle. 
# Unsloth will compile/download llama.cpp internally and output the merged F16 GGUF file.
#model.save_pretrained_gguf(
#    "/kaggle/working/spurgeon_f16_gguf",
#    tokenizer,
#    quantization_method = "f16",
#)

## 7. Upload GGUF Model to Hugging Face Hub

We securely load your Hugging Face write token from Kaggle Secrets, initialize the Hugging Face API, and upload the 6GB `unsloth.F16.gguf` file directly to your model repository.

In [7]:
# Upload GGUF file to Hugging Face Hub using Kaggle Secrets
#from huggingface_hub import HfApi, login
#from kaggle_secrets import UserSecretsClient
#import glob
#import os

#try:
    # Load Hugging Face API write token from Kaggle Secrets (Add 'HF_TOKEN' in Add-ons -> Secrets)
#    user_secrets = UserSecretsClient()
#    hf_token = user_secrets.get_secret("HF_TOKEN")
#    login(token=hf_token)
    
#    api = HfApi()
#    repo_id = "rafaelvieirar1r/qwen2.5-3b-spurgeon-gguf-phase1"
    
#    print(f"Creating Hugging Face repository {repo_id} (if it does not exist)...")
#    api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
    
    # Dynamically find any .gguf file under /kaggle/working/
#    gguf_files = glob.glob("/kaggle/working/**/*.gguf", recursive=True)
#    if not gguf_files:
#        raise FileNotFoundError("No GGUF file found under /kaggle/working")
    
#    file_path = gguf_files[0]
#    file_name = os.path.basename(file_path)
    #file_path = "/kaggle/working/spurgeon_f16_gguf_gguf/qwen2.5-3b.F16.gguf"

#    print(f"Found GGUF file: {file_path}")
#    print(f"Uploading {file_name} to Hugging Face Hub...")
#    api.upload_file(
#        path_or_fileobj=file_path,
#        path_in_repo=file_name,
#        repo_id=repo_id,
#        repo_type="model"
#    )
#    print("Model successfully uploaded to Hugging Face!")
#except Exception as e:
#    print(f"Failed to upload to Hugging Face: {e}")
#    print("Please ensure you have toggled Internet ON and added the 'HF_TOKEN' key to Kaggle Secrets.")